# Showing Unicode Characters in a Table

In the lecture, we have defined the *Unicode alphabet* as the set
$$ \Sigma_{\textrm{Unicode}} = \{ 0, 1, \cdots, 1\,114\,111 \}, $$
i.e. every character is identified with a natural number, its *code point*.  Code points are usually
written in hexadecimal notation using the prefix `U+`, so the largest code point is `U+10FFFF`.

This notebook implements a small program that displays a range of Unicode characters in a table.  For
every character, the table shows both the *glyph*, i.e. the picture of the character, and its code point.
The official Unicode name of a character is shown when the mouse pointer rests on its glyph.

## Imports

- The module `unicodedata` gives access to the *Unicode Character Database*.  We use it to look up the
  name and the *general category* of a character.
- The module `html` provides the function `html.escape`, which we need since characters like `<` and `&`
  have a special meaning in HTML.
- The function `display` and the class `HTML` from the module `IPython.display` enable us to render an
  HTML string as a table inside the notebook.

In [1]:
import unicodedata
import html
from IPython.display import display, HTML

## Printable Characters

Not every code point can be displayed:
- Many code points are not yet assigned to a character.
- Some code points are *control codes*, e.g. the code point $10$ encodes a line feed.
- Other code points are reserved for special purposes, e.g. the *surrogates* in the range from `U+D800`
  to `U+DFFF`, which are used by the encoding UTF-16.

Every character belongs to a *general category*, which is a string consisting of two letters.  The first
letter gives the major class of the character, e.g. `L` for letters, `N` for numbers, `P` for
punctuation, and `C` for *other* characters.  All code points that cannot be displayed belong to one of the
categories `Cc` (control), `Cf` (format), `Cs` (surrogate), `Co` (private use), and `Cn` (unassigned).
Besides these, the space characters of category `Zs` are displayed as nothing.

The function `is_printable(c)` checks whether the code point `c` stands for a character that has a visible
glyph.  The function `chr(c)` converts the code point `c` into a string of length $1$, while the function
`unicodedata.category` returns the general category of this character.

In [2]:
def is_printable(c):
    category = unicodedata.category(chr(c))
    return category[0] != 'C' and category != 'Zs'

Let's test this function with the code points of the letter `A`, the space character, and the line feed.

In [3]:
is_printable(65), is_printable(32), is_printable(10)

(True, False, False)

## Formatting a Single Cell

The function `cell(c)` returns the HTML code of the two table cells that describe the code point `c`.
- The first cell contains the glyph.  The glyph is escaped with `html.escape`, and the official name of
  the character, which is returned by `unicodedata.name`, is attached as the attribute `title`.  Browsers
  show this attribute as a *tooltip*.  If the character is not printable, the cell is left empty and gets a
  grey background instead.  We do not use a placeholder symbol since every symbol we could choose is itself a
  Unicode character and might therefore occur in the table.  The second argument of `unicodedata.name` is returned when a character has no name, which is
  the case, e.g., for control codes.
- The second cell contains the code point in the format `U+XXXX`.  The format specification `04X` writes
  the number `c` in hexadecimal notation with capital letters, using at least $4$ digits.

In [4]:
def cell(c):
    name = unicodedata.name(chr(c), 'no name')
    if is_printable(c):
        glyph, style = html.escape(chr(c)), ''
    else:
        glyph, style = '', 'background:lightgrey; '
    return (f'<td title="{html.escape(name)}" style="{style}font-size:150%; text-align:center">{glyph}</td>'
            f'<td style="font-family:monospace">U+{c:04X}</td>')

## Building the Table

The function `unicode_table(first, last, columns)` returns an HTML table that shows all characters with code
points from `first` up to and including `last`.  Every row of the table shows `columns` characters.
- `codes` is the list of all code points that are to be shown.
- The header of the table contains the column titles `Glyph` and `Code` once for every character in a row.
- The list comprehension that computes `rows` splits the list `codes` into slices of length `columns`.
  Every slice is turned into one row of the table.  The last row might be shorter than the other rows.

In [5]:
def unicode_table(first, last, columns=8):
    codes  = list(range(first, last + 1))
    header = '<tr>' + '<th>Glyph</th><th>Code</th>' * columns + '</tr>'
    rows   = [ '<tr>' + ''.join(cell(c) for c in codes[i:i+columns]) + '</tr>'
               for i in range(0, len(codes), columns)
             ]
    return HTML('<table>' + header + ''.join(rows) + '</table>')

## Trying it Out

We start with the printable characters of the *ASCII alphabet*, which are the characters with the code
points from $32$ up to $126$.  Compare this table with Table 1.1 of the lecture notes.

In [6]:
display(unicode_table(32, 126))

Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code
,U+0020,!,U+0021,"""",U+0022,#,U+0023,$,U+0024,%,U+0025,&,U+0026,',U+0027
(,U+0028,),U+0029,*,U+002A,+,U+002B,",",U+002C,-,U+002D,.,U+002E,/,U+002F
0,U+0030,1,U+0031,2,U+0032,3,U+0033,4,U+0034,5,U+0035,6,U+0036,7,U+0037
8,U+0038,9,U+0039,:,U+003A,;,U+003B,<,U+003C,=,U+003D,>,U+003E,?,U+003F
@,U+0040,A,U+0041,B,U+0042,C,U+0043,D,U+0044,E,U+0045,F,U+0046,G,U+0047
H,U+0048,I,U+0049,J,U+004A,K,U+004B,L,U+004C,M,U+004D,N,U+004E,O,U+004F
P,U+0050,Q,U+0051,R,U+0052,S,U+0053,T,U+0054,U,U+0055,V,U+0056,W,U+0057
X,U+0058,Y,U+0059,Z,U+005A,[,U+005B,\,U+005C,],U+005D,^,U+005E,_,U+005F
`,U+0060,a,U+0061,b,U+0062,c,U+0063,d,U+0064,e,U+0065,f,U+0066,g,U+0067
h,U+0068,i,U+0069,j,U+006A,k,U+006B,l,U+006C,m,U+006D,n,U+006E,o,U+006F


The Greek letters start at the code point `U+0391`.  Note that there is no character with the code point
`U+03A2`: this code point is not assigned, so its cell is grey.  The lower case letter `ς` at `U+03C2` is the variant of `σ` that
is used at the end of a word, so there is no need for a corresponding capital letter.

In [7]:
display(unicode_table(0x0391, 0x03C9))

Unicode also contains many mathematical symbols.  Some of the most important ones are found in the block
*Mathematical Operators*, which starts at the code point `U+2200`.

In [8]:
display(unicode_table(0x2200, 0x225F))

Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code
∀,U+2200,∁,U+2201,∂,U+2202,∃,U+2203,∄,U+2204,∅,U+2205,∆,U+2206,∇,U+2207
∈,U+2208,∉,U+2209,∊,U+220A,∋,U+220B,∌,U+220C,∍,U+220D,∎,U+220E,∏,U+220F
∐,U+2210,∑,U+2211,−,U+2212,∓,U+2213,∔,U+2214,∕,U+2215,∖,U+2216,∗,U+2217
∘,U+2218,∙,U+2219,√,U+221A,∛,U+221B,∜,U+221C,∝,U+221D,∞,U+221E,∟,U+221F
∠,U+2220,∡,U+2221,∢,U+2222,∣,U+2223,∤,U+2224,∥,U+2225,∦,U+2226,∧,U+2227
∨,U+2228,∩,U+2229,∪,U+222A,∫,U+222B,∬,U+222C,∭,U+222D,∮,U+222E,∯,U+222F
∰,U+2230,∱,U+2231,∲,U+2232,∳,U+2233,∴,U+2234,∵,U+2235,∶,U+2236,∷,U+2237
∸,U+2238,∹,U+2239,∺,U+223A,∻,U+223B,∼,U+223C,∽,U+223D,∾,U+223E,∿,U+223F
≀,U+2240,≁,U+2241,≂,U+2242,≃,U+2243,≄,U+2244,≅,U+2245,≆,U+2246,≇,U+2247
≈,U+2248,≉,U+2249,≊,U+224A,≋,U+224B,≌,U+224C,≍,U+224D,≎,U+224E,≏,U+224F


The *CJK Unified Ideographs* are Chinese characters, which are also used in Japanese and Korean.  This block
starts at `U+4E00`.  The first of these characters is `一`, the Chinese character for the number $1$.

In [9]:
display(unicode_table(0x4E00, 0x4E3F))

Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code
一,U+4E00,丁,U+4E01,丂,U+4E02,七,U+4E03,丄,U+4E04,丅,U+4E05,丆,U+4E06,万,U+4E07
丈,U+4E08,三,U+4E09,上,U+4E0A,下,U+4E0B,丌,U+4E0C,不,U+4E0D,与,U+4E0E,丏,U+4E0F
丐,U+4E10,丑,U+4E11,丒,U+4E12,专,U+4E13,且,U+4E14,丕,U+4E15,世,U+4E16,丗,U+4E17
丘,U+4E18,丙,U+4E19,业,U+4E1A,丛,U+4E1B,东,U+4E1C,丝,U+4E1D,丞,U+4E1E,丟,U+4E1F
丠,U+4E20,両,U+4E21,丢,U+4E22,丣,U+4E23,两,U+4E24,严,U+4E25,並,U+4E26,丧,U+4E27
丨,U+4E28,丩,U+4E29,个,U+4E2A,丫,U+4E2B,丬,U+4E2C,中,U+4E2D,丮,U+4E2E,丯,U+4E2F
丰,U+4E30,丱,U+4E31,串,U+4E32,丳,U+4E33,临,U+4E34,丵,U+4E35,丶,U+4E36,丷,U+4E37
丸,U+4E38,丹,U+4E39,为,U+4E3A,主,U+4E3B,丼,U+4E3C,丽,U+4E3D,举,U+4E3E,丿,U+4E3F


Unicode also contains a large number of *emojis*.  The block *Emoticons* starts at `U+1F600`.  Note
that these code points are bigger than $2^{16} = 65\,536$, so they need more than $4$ hexadecimal digits.

In [10]:
display(unicode_table(0x1F600, 0x1F64F))

Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code
😀,U+1F600,😁,U+1F601,😂,U+1F602,😃,U+1F603,😄,U+1F604,😅,U+1F605,😆,U+1F606,😇,U+1F607
😈,U+1F608,😉,U+1F609,😊,U+1F60A,😋,U+1F60B,😌,U+1F60C,😍,U+1F60D,😎,U+1F60E,😏,U+1F60F
😐,U+1F610,😑,U+1F611,😒,U+1F612,😓,U+1F613,😔,U+1F614,😕,U+1F615,😖,U+1F616,😗,U+1F617
😘,U+1F618,😙,U+1F619,😚,U+1F61A,😛,U+1F61B,😜,U+1F61C,😝,U+1F61D,😞,U+1F61E,😟,U+1F61F
😠,U+1F620,😡,U+1F621,😢,U+1F622,😣,U+1F623,😤,U+1F624,😥,U+1F625,😦,U+1F626,😧,U+1F627
😨,U+1F628,😩,U+1F629,😪,U+1F62A,😫,U+1F62B,😬,U+1F62C,😭,U+1F62D,😮,U+1F62E,😯,U+1F62F
😰,U+1F630,😱,U+1F631,😲,U+1F632,😳,U+1F633,😴,U+1F634,😵,U+1F635,😶,U+1F636,😷,U+1F637
😸,U+1F638,😹,U+1F639,😺,U+1F63A,😻,U+1F63B,😼,U+1F63C,😽,U+1F63D,😾,U+1F63E,😿,U+1F63F
🙀,U+1F640,🙁,U+1F641,🙂,U+1F642,🙃,U+1F643,🙄,U+1F644,🙅,U+1F645,🙆,U+1F646,🙇,U+1F647
🙈,U+1F648,🙉,U+1F649,🙊,U+1F64A,🙋,U+1F64B,🙌,U+1F64C,🙍,U+1F64D,🙎,U+1F64E,🙏,U+1F64F


## Cuneiform

Finally, Unicode also covers historical scripts.  *Cuneiform* is one of the oldest known writing systems.  It
was used in Mesopotamia for more than 3000 years, starting around 3400 BC.  The characters were pressed into
clay tablets with a reed stylus, which gives them their wedge-shaped look (Latin *cuneus* means *wedge*).  The
block *Cuneiform* starts at `U+12000`.  The next cell shows the first $100$ characters of this block.  Their
names, e.g. `CUNEIFORM SIGN A`, are shown as tooltips.

**Note:** Your browser needs a font that contains these characters, e.g. *Noto Sans Cuneiform*.  Otherwise,
the characters are shown as empty boxes.

In [11]:
display(unicode_table(0x12000, 0x12063))

Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code,Glyph,Code
𒀀,U+12000,𒀁,U+12001,𒀂,U+12002,𒀃,U+12003,𒀄,U+12004,𒀅,U+12005,𒀆,U+12006,𒀇,U+12007
𒀈,U+12008,𒀉,U+12009,𒀊,U+1200A,𒀋,U+1200B,𒀌,U+1200C,𒀍,U+1200D,𒀎,U+1200E,𒀏,U+1200F
𒀐,U+12010,𒀑,U+12011,𒀒,U+12012,𒀓,U+12013,𒀔,U+12014,𒀕,U+12015,𒀖,U+12016,𒀗,U+12017
𒀘,U+12018,𒀙,U+12019,𒀚,U+1201A,𒀛,U+1201B,𒀜,U+1201C,𒀝,U+1201D,𒀞,U+1201E,𒀟,U+1201F
𒀠,U+12020,𒀡,U+12021,𒀢,U+12022,𒀣,U+12023,𒀤,U+12024,𒀥,U+12025,𒀦,U+12026,𒀧,U+12027
𒀨,U+12028,𒀩,U+12029,𒀪,U+1202A,𒀫,U+1202B,𒀬,U+1202C,𒀭,U+1202D,𒀮,U+1202E,𒀯,U+1202F
𒀰,U+12030,𒀱,U+12031,𒀲,U+12032,𒀳,U+12033,𒀴,U+12034,𒀵,U+12035,𒀶,U+12036,𒀷,U+12037
𒀸,U+12038,𒀹,U+12039,𒀺,U+1203A,𒀻,U+1203B,𒀼,U+1203C,𒀽,U+1203D,𒀾,U+1203E,𒀿,U+1203F
𒁀,U+12040,𒁁,U+12041,𒁂,U+12042,𒁃,U+12043,𒁄,U+12044,𒁅,U+12045,𒁆,U+12046,𒁇,U+12047
𒁈,U+12048,𒁉,U+12049,𒁊,U+1204A,𒁋,U+1204B,𒁌,U+1204C,𒁍,U+1204D,𒁎,U+1204E,𒁏,U+1204F
